# SC-2-Functions-State - Fonctions et Etat

**Navigation** : [Index](../README.md) | [<< Basics](SC-1-Solidity-Basics.ipynb) | [Inheritance >>](SC-3-Inheritance.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les **data locations** (storage, memory, calldata)
2. Maitriser la **visibilite** des fonctions
3. Utiliser les **modifiers** (view, pure, payable)

### Duree estimee : 45 minutes

---

## 1. Data Locations

Solidite definit trois emplacements de donnees :

| Location | Description | Cout gas |
|----------|-------------|----------|
| `storage` | Blockchain permanente | Eleve |
| `memory` | Temporaire (RAM) | Faible |
| `calldata` | Donnees d'entree (read-only) | Minimal |

In [1]:
# Data locations en Solidity
DATA_LOCATION_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract DataLocations {
    // STORAGE : stocke sur la blockchain (permanent)
    uint256[] public numbers;  // storage par defaut pour variables d'etat
    mapping(address => uint256) public balances;

    // MEMORY : temporaire, efface apres l'appel
    function processArray(uint256[] memory arr) public pure returns (uint256) {
        uint256 sum = 0;
        for (uint i = 0; i < arr.length; i++) {
            sum += arr[i];
        }
        return sum;
    }

    // CALLDATA : read-only, le moins couteux
    function getElement(uint256[] calldata arr, uint256 index) 
        external pure returns (uint256) 
    {
        return arr[index];
    }

    // Storage reference pour modification
    function updateNumber(uint256 index, uint256 value) public {
        numbers[index] = value;  // Modification directe du storage
    }
}
'''

print("Data Locations en Solidity:")
print(DATA_LOCATION_EXAMPLE)

Data Locations en Solidity:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract DataLocations {
    // STORAGE : stocke sur la blockchain (permanent)
    uint256[] public numbers;  // storage par defaut pour variables d'etat
    mapping(address => uint256) public balances;

    // MEMORY : temporaire, efface apres l'appel
    function processArray(uint256[] memory arr) public pure returns (uint256) {
        uint256 sum = 0;
        for (uint i = 0; i < arr.length; i++) {
            sum += arr[i];
        }
        return sum;
    }

    // CALLDATA : read-only, le moins couteux
    function getElement(uint256[] calldata arr, uint256 index) 
        external pure returns (uint256) 
    {
        return arr[index];
    }

    // Storage reference pour modification
    function updateNumber(uint256 index, uint256 value) public {
        numbers[index] = value;  // Modification directe du storage
    }
}



### 1.1 Regles de data location

- **Variables d'etat** : toujours en `storage`
- **Arguments de fonction** : `memory` ou `calldata` pour les types complexes
- **Variables locales** : `storage` (reference) ou `memory` (valeur copiee)

In [2]:
# Storage vs Memory : exemple critique
STORAGE_MEMORY_EXAMPLE = '''
contract StorageVsMemory {
    struct User {
        string name;
        uint256 balance;
    }

    User[] public users;

    // ATTENTION : storage reference vs memory copy
    function updateBalanceGood(uint256 index, uint256 amount) public {
        // storage : modifie directement le tableau
        User storage user = users[index];
        user.balance = amount;
    }

    function updateBalanceBad(uint256 index, uint256 amount) public {
        // memory : copie locale, NE MODIFIE PAS le storage
        User memory user = users[index];
        user.balance = amount;  // Changement local uniquement!
        // Pour sauvegarder: users[index] = user;
    }

    function addUser(string memory name, uint256 balance) public {
        users.push(User(name, balance));
    }
}
'''

print("Attention a la difference storage/memory:")
print(STORAGE_MEMORY_EXAMPLE)

Attention a la difference storage/memory:

contract StorageVsMemory {
    struct User {
        string name;
        uint256 balance;
    }

    User[] public users;

    // ATTENTION : storage reference vs memory copy
    function updateBalanceGood(uint256 index, uint256 amount) public {
        // storage : modifie directement le tableau
        User storage user = users[index];
        user.balance = amount;
    }

    function updateBalanceBad(uint256 index, uint256 amount) public {
        // memory : copie locale, NE MODIFIE PAS le storage
        User memory user = users[index];
        user.balance = amount;  // Changement local uniquement!
        // Pour sauvegarder: users[index] = user;
    }

    function addUser(string memory name, uint256 balance) public {
        users.push(User(name, balance));
    }
}



---

## 2. Visibilite des fonctions

| Visibilite | Exterieur | Contrat enfant | Meme contrat |
|------------|-----------|----------------|--------------|
| `public` | Oui | Oui | Oui |
| `external` | Oui | Non | Non |
| `internal` | Non | Oui | Oui |
| `private` | Non | Non | Oui |

In [3]:
# Visibilite des fonctions
VISIBILITY_EXAMPLE = '''
contract VisibilityExample {
    uint256 private privateVar = 1;
    uint256 internal internalVar = 2;
    uint256 public publicVar = 3;

    // PUBLIC : accessible partout
    function publicFunction() public pure returns (string memory) {
        return "Je suis publique!";
    }

    // EXTERNAL : uniquement depuis l'exterieur
    function externalFunction() external pure returns (string memory) {
        return "Je suis externe!";
    }

    // INTERNAL : ce contrat et les enfants
    function internalFunction() internal pure returns (string memory) {
        return "Je suis interne!";
    }

    // PRIVATE : uniquement ce contrat
    function privateFunction() private pure returns (string memory) {
        return "Je suis prive!";
    }

    // Appel interne possible
    function callInternal() public pure returns (string memory) {
        return internalFunction();  // OK
    }

    // ERREUR : externalFunction() pas appelable en interne
    // function callExternal() public pure returns (string memory) {
    //     return externalFunction();  // ERREUR!
    // }
}
'''

print("Visibilite des fonctions:")
print(VISIBILITY_EXAMPLE)

Visibilite des fonctions:

contract VisibilityExample {
    uint256 private privateVar = 1;
    uint256 internal internalVar = 2;
    uint256 public publicVar = 3;

    // PUBLIC : accessible partout
    function publicFunction() public pure returns (string memory) {
        return "Je suis publique!";
    }

    // EXTERNAL : uniquement depuis l'exterieur
    function externalFunction() external pure returns (string memory) {
        return "Je suis externe!";
    }

    // INTERNAL : ce contrat et les enfants
    function internalFunction() internal pure returns (string memory) {
        return "Je suis interne!";
    }

    // PRIVATE : uniquement ce contrat
    function privateFunction() private pure returns (string memory) {
        return "Je suis prive!";
    }

    // Appel interne possible
    function callInternal() public pure returns (string memory) {
        return internalFunction();  // OK
    }

    // ERREUR : externalFunction() pas appelable en interne
    // function

In [4]:
# External vs Public : optimisation gas
EXTERNAL_OPTIMIZATION = '''
contract GasOptimization {
    // EXTERNAL est moins couteux pour les tableaux
    // car les donnees restent en calldata (pas de copie)

    // Moins efficace (copie en memory)
    function sumPublic(uint256[] memory arr) public pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }

    // Plus efficace (reste en calldata)
    function sumExternal(uint256[] calldata arr) external pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }
}
'''

print("Optimisation gas avec external + calldata:")
print(EXTERNAL_OPTIMIZATION)

Optimisation gas avec external + calldata:

contract GasOptimization {
    // EXTERNAL est moins couteux pour les tableaux
    // car les donnees restent en calldata (pas de copie)

    // Moins efficace (copie en memory)
    function sumPublic(uint256[] memory arr) public pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }

    // Plus efficace (reste en calldata)
    function sumExternal(uint256[] calldata arr) external pure returns (uint256) {
        uint256 total = 0;
        for (uint i = 0; i < arr.length; i++) {
            total += arr[i];
        }
        return total;
    }
}



---

## 3. Modificateurs de fonction

### 3.1 view et pure

In [5]:
# view et pure
VIEW_PURE_EXAMPLE = '''
contract ViewPureExample {
    uint256 public value = 42;

    // VIEW : lit le storage mais ne modifie pas
    function getValue() public view returns (uint256) {
        return value;  // Lecture OK
        // value = 100;  // ERREUR : ecriture interdite
    }

    // PURE : ni lecture ni ecriture du storage
    function add(uint256 a, uint256 b) public pure returns (uint256) {
        return a + b;  // Aucun acces au storage
        // return value;  // ERREUR : lecture interdite
    }

    // Fonction normale : peut lire et ecrire
    function increment() public returns (uint256) {
        value += 1;  // Ecriture OK
        return value;
    }
}
'''

print("view = lecture seule, pure = sans acces storage:")
print(VIEW_PURE_EXAMPLE)

view = lecture seule, pure = sans acces storage:

contract ViewPureExample {
    uint256 public value = 42;

    // VIEW : lit le storage mais ne modifie pas
    function getValue() public view returns (uint256) {
        return value;  // Lecture OK
        // value = 100;  // ERREUR : ecriture interdite
    }

    // PURE : ni lecture ni ecriture du storage
    function add(uint256 a, uint256 b) public pure returns (uint256) {
        return a + b;  // Aucun acces au storage
        // return value;  // ERREUR : lecture interdite
    }

    // Fonction normale : peut lire et ecrire
    function increment() public returns (uint256) {
        value += 1;  // Ecriture OK
        return value;
    }
}



### 3.2 payable

In [6]:
# payable pour recevoir des ETH
PAYABLE_EXAMPLE = '''
contract PayableExample {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    // Fonction payable : peut recevoir des ETH
    function deposit() public payable {
        // msg.value contient le montant envoye en wei
        balances[msg.sender] += msg.value;
    }

    // Sans payable : revert si ETH envoye
    function noEther() public pure returns (string memory) {
        return "Je n'accepte pas l'ETH";
    }

    // Retirer des ETH
    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Fonds insuffisants");
        balances[msg.sender] -= amount;
        (bool success, ) = payable(msg.sender).call{value: amount}("");
        require(success, "Echec du transfert");
    }

    // Receive : appele lors d'un transfert ETH simple
    receive() external payable {
        balances[msg.sender] += msg.value;
    }

    // Fallback : appele pour les appels inconnus
    fallback() external payable {
        // Optionnel : logique de fallback
    }

    // Consulter le solde du contrat
    function getContractBalance() public view returns (uint256) {
        return address(this).balance;
    }
}
'''

print("payable permet de recevoir des ETH:")
print(PAYABLE_EXAMPLE)

payable permet de recevoir des ETH:

contract PayableExample {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    // Fonction payable : peut recevoir des ETH
    function deposit() public payable {
        // msg.value contient le montant envoye en wei
        balances[msg.sender] += msg.value;
    }

    // Sans payable : revert si ETH envoye
    function noEther() public pure returns (string memory) {
        return "Je n'accepte pas l'ETH";
    }

    // Retirer des ETH
    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Fonds insuffisants");
        balances[msg.sender] -= amount;
        (bool success, ) = payable(msg.sender).call{value: amount}("");
        require(success, "Echec du transfert");
    }

    // Receive : appele lors d'un transfert ETH simple
    receive() external payable {
        balances[msg.sender] += msg.value;
    }

    // Fallback

---

## 4. Custom Modifiers

Les modifiers permettent de reutiliser des conditions prealables.

In [7]:
# Custom modifiers
MODIFIER_EXAMPLE = '''
contract ModifierExample {
    address public owner;
    bool public paused = false;

    constructor() {
        owner = msg.sender;
    }

    // Modifier : seul le proprietaire
    modifier onlyOwner() {
        require(msg.sender == owner, "Pas le proprietaire");
        _;  // Continue l'execution de la fonction
    }

    // Modifier : contrat non en pause
    modifier whenNotPaused() {
        require(!paused, "Contrat en pause");
        _;
    }

    // Utilisation des modifiers
    function pause() public onlyOwner {
        paused = true;
    }

    function unpause() public onlyOwner {
        paused = false;
    }

    function sensitiveOperation() public whenNotPaused {
        // Cette fonction ne peut pas etre appelee si paused = true
    }

    // Modifiers chaines
    function criticalOperation() public onlyOwner whenNotPaused {
        // Les deux conditions sont verifiees
    }
}
'''

print("Custom modifiers pour conditions reutilisables:")
print(MODIFIER_EXAMPLE)

Custom modifiers pour conditions reutilisables:

contract ModifierExample {
    address public owner;
    bool public paused = false;

    constructor() {
        owner = msg.sender;
    }

    // Modifier : seul le proprietaire
    modifier onlyOwner() {
        require(msg.sender == owner, "Pas le proprietaire");
        _;  // Continue l'execution de la fonction
    }

    // Modifier : contrat non en pause
    modifier whenNotPaused() {
        require(!paused, "Contrat en pause");
        _;
    }

    // Utilisation des modifiers
    function pause() public onlyOwner {
        paused = true;
    }

    function unpause() public onlyOwner {
        paused = false;
    }

    function sensitiveOperation() public whenNotPaused {
        // Cette fonction ne peut pas etre appelee si paused = true
    }

    // Modifiers chaines
    function criticalOperation() public onlyOwner whenNotPaused {
        // Les deux conditions sont verifiees
    }
}



---

## 5. Fonctions speciales

In [8]:
# Constructor, receive, fallback
SPECIAL_FUNCTIONS = '''
contract SpecialFunctions {
    address public owner;
    string public name;

    // CONSTRUCTOR : execute une seule fois au deploiement
    constructor(string memory _name) {
        owner = msg.sender;
        name = _name;
    }

    // RECEIVE : appele pour les transferts ETH simples
    // (sans data, msg.data vide)
    receive() external payable {
        emit Received(msg.sender, msg.value);
    }

    // FALLBACK : appele pour les appels inconnus
    // (avec data, ou si receive n'existe pas)
    fallback() external payable {
        emit FallbackCalled(msg.sender, msg.value, msg.data);
    }

    event Received(address from, uint256 amount);
    event FallbackCalled(address from, uint256 amount, bytes data);
}
'''

print("Fonctions speciales:")
print(SPECIAL_FUNCTIONS)

Fonctions speciales:

contract SpecialFunctions {
    address public owner;
    string public name;

    // CONSTRUCTOR : execute une seule fois au deploiement
    constructor(string memory _name) {
        owner = msg.sender;
        name = _name;
    }

    // RECEIVE : appele pour les transferts ETH simples
    // (sans data, msg.data vide)
    receive() external payable {
        emit Received(msg.sender, msg.value);
    }

    // FALLBACK : appele pour les appels inconnus
    // (avec data, ou si receive n'existe pas)
    fallback() external payable {
        emit FallbackCalled(msg.sender, msg.value, msg.data);
    }

    event Received(address from, uint256 amount);
    event FallbackCalled(address from, uint256 amount, bytes data);
}



---

## 6. Exercices

### Exercice 1 : Contrat Bank

Creez un contrat qui permet de deposer et retirer des ETH avec un modifier `onlyOwner`.

In [ ]:
# Exercice 1 : Contrat Bank
EXERCICE_BANK = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract Bank {
    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        // TODO: Initialiser le proprietaire
    }

    modifier onlyOwner() {
        // TODO: Verifier que msg.sender est le proprietaire
        // TODO: Utiliser require et ne pas oublier le _;
        _;
    }

    function deposit() public payable {
        // TODO: Ajouter msg.value au solde de msg.sender
    }

    function withdraw(uint256 amount) public {
        // TODO: Verifier que l'utilisateur a assez de fonds
        // TODO: Reduire le solde de l'utilisateur
        // TODO: Transferer les ETH a l'utilisateur
    }

    function emergencyWithdraw() public onlyOwner {
        // TODO: Transferer tout le solde du contrat au proprietaire
    }
}
'''
print("Squelette Bank:")
print(EXERCICE_BANK)

### Exercice 2 : Contrat Pausable

Creez un contrat avec des fonctions qui peuvent etre mises en pause.

In [ ]:
# Exercice 2 : Contrat Pausable
EXERCICE_PAUSABLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

contract Pausable {
    address public owner;
    bool public paused = false;
    uint256 public counter = 0;

    constructor() {
        // TODO: Initialiser le proprietaire
    }

    modifier onlyOwner() {
        // TODO: Verifier que msg.sender est le proprietaire
        _;
    }

    modifier whenNotPaused() {
        // TODO: Verifier que le contrat n est pas en pause
        _;
    }

    function pause() public onlyOwner {
        // TODO: Mettre le contrat en pause
    }

    function unpause() public onlyOwner {
        // TODO: Reprendre le contrat
    }

    function increment() public whenNotPaused {
        // TODO: Incrementer le compteur
    }

    function reset() public onlyOwner whenNotPaused {
        // TODO: Remettre le compteur a zero
    }
}
'''
print("Squelette Pausable:")
print(EXERCICE_PAUSABLE)

---

## 7. Resume

| Concept | Description |
|---------|-------------|
| `storage` | Donnees permanentes sur blockchain |
| `memory` | Donnees temporaires en RAM |
| `calldata` | Donnees d'entree read-only |
| `public` | Accessible partout |
| `external` | Accessible uniquement de l'exterieur |
| `internal` | Ce contrat et enfants |
| `private` | Uniquement ce contrat |
| `view` | Lecture seule du storage |
| `pure` | Aucun acces au storage |
| `payable` | Peut recevoir des ETH |

---

**Notebook suivant** : [SC-3-Inheritance](SC-3-Inheritance.ipynb)